# Biblical Muse Glimmer 30B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** `meta-models/Muse-Glimmer-30B` — Meta's own release of Muse Glimmer 30B (`MuseGlimmerForConditionalGeneration`, dense causal transformer + ViT perception encoder, Apache-2.0, ungated), bf16, 59.6 GB. Unsloth's model mapper would normally swap this name for its re-packed `unsloth/Muse-Glimmer-30B-unsloth-bnb-4bit`; every `from_pretrained` here passes `use_exact_model_name=True` so the weights, tokenizer and chat template all come from Meta's repo and nowhere else. The 4-bit (nf4) quantization happens on the fly at load time. The same repo serves the optional GGUF export in Section 13, so there is one download, not two.

**Dataset:** Per-persona datagen JSONL — each persona has its own system prompt and distinctive voice

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Chat Template:** Tokenizer-native Muse Glimmer template (`<|start|>role<|message|>content<|eot|>`; assistant answers render on the `to=user` channel), applied via `tokenizer.apply_chat_template`

**Reasoning strength:** Glimmer has no thinking on/off switch. Its template *always* appends `Reasoning strength: <level>.` to the system prompt (`low` / `medium` / `high` / `xhigh`, default `high`). Training data here is plain answers with no reasoning channel, so formatting, the eval cells and the GGUF export all use `REASONING_STRENGTH` from Cell 1 (`low`).

This notebook is modeled on `qwen_38_27b/biblical_qwen3_8_27b_sft_unsloth_4bit.ipynb`, which produced a working adapter on this machine. Differences from it are Glimmer-specific and commented where they occur. There is no DPO counterpart yet.

## 1. Configuration

In [1]:
import os

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
if os.path.exists("/workspace/training/biblical"):
    PROJECT_ROOT = "/workspace/training/biblical"
elif os.path.exists("/workspace/biblical"):
    PROJECT_ROOT = "/workspace/biblical"
else:
    PROJECT_ROOT = "/home/spark/projects/training/biblical"

OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== HUGGING FACE CACHE ===========================
# Use the default Hub cache: /root/.cache/huggingface/hub in the unsloth-notebook container,
# which is bind-mounted from /home/spark/.cache/huggingface/hub on the host. Do NOT override
# HF_HUB_CACHE: pointing it at the cache root (not hub/) made unsloth build a second
# models--* tree beside hub/ and re-download every base model.

# =========================== MODEL CONFIGURATION ===========================
# meta-models/Muse-Glimmer-30B is Meta's own bf16 release (59.6 GB, MuseGlimmerForConditionalGeneration,
# model_type muse_glimmer). Unsloth's mapper (unsloth/models/mapper.py, checked in 2026.9.7) lists this
# name under unsloth/Muse-Glimmer-30B-unsloth-bnb-4bit and would silently load that re-pack instead.
# Every from_pretrained in this notebook passes use_exact_model_name=True (loader.py: "Only the mapper
# substitution answers to use_exact_model_name"), so Meta's repo is what gets loaded and bnb nf4
# quantization is applied on the fly. The model-load cell prints the name that was actually resolved.
BASE_LLM = "meta-models/Muse-Glimmer-30B"
MODEL_NAME_BASE = "biblical_muse_glimmer_30b_unsloth_4bit"

# =========================== REASONING STRENGTH ===========================
# Glimmer's chat template has NO enable_thinking. It unconditionally appends
#   "\n\nReasoning strength: <level>.\n\n# Valid recipients: \"self\", \"user\"."
# to the system prompt, with <level> taken from the template variable `reasoning_strength`
# (low / medium / high / xhigh; the template default is 'high'). There is no "off".
#
# The training data is plain persona answers with no reasoning channel (`to=self`), so the
# lowest level is used, and the SAME value is passed everywhere a prompt is rendered:
# training formatting, both inference cells, and the template default baked into the GGUF.
# Serving this adapter with a different level puts a system-prompt line in front of the model
# that training never showed it.
REASONING_STRENGTH = "low"
if REASONING_STRENGTH not in ("low", "medium", "high", "xhigh"):
    raise ValueError(f"REASONING_STRENGTH must be low/medium/high/xhigh, got {REASONING_STRENGTH!r}")
REASONING_MARKER = f"Reasoning strength: {REASONING_STRENGTH}."

# =========================== INPUT DATA ===========================
# Combined multi-turn ShareGPT JSONL from the datagen notebooks (per-persona + augmented).
# Already quality-filtered, multi-turn (4 QA pairs per conversation), grouped by topic.
INPUT_DATA_FILE = f"{PROJECT_ROOT}/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl"

# =========================== PERSONA SYSTEM PROMPTS ===========================
# System prompts are EXTRACTED from the JSONL at load time (see the data loading cell).
# This keeps training in sync with datagen — regenerate data with new/changed prompts and
# this notebook picks them up automatically. After loading, `persona_system_prompts` maps
# persona_key -> full prompt text, and is saved alongside the LoRA adapters for inference.

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
GGUF_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/gguf"

# =========================== TRAINING HYPERPARAMETERS ===========================
# MAX_SEQ_LENGTH: measured on 2026-11-09 with the Glimmer tokenizer and this template at
# reasoning_strength="low": 4,352 conversations, median 1,588 tokens, max 3,033. So 3072
# fits every row with 39 tokens to spare. The formatting cell re-measures and RAISES if any
# row would be truncated - truncation cuts the final answer and its <|eot|>.
MAX_SEQ_LENGTH = 3072
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
TARGET_EPOCHS = 1
SFT_MAX_EXAMPLES = 0  # 0 = use all valid conversations; set e.g. 500/1000/3000 for capped runs

# =========================== CHECKPOINTING ===========================
# One epoch is 544 optimizer steps at batch 2 x accum 4 and runs for hours. Checkpoint often
# enough that a crash costs minutes instead of a day; the training cell auto-resumes.
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3

# gc.collect() + torch.cuda.empty_cache() cadence during training, and the cadence at which
# allocated/reserved memory is printed. Aligned with the checkpoint interval.
CLEANUP_STEPS = 50

# =========================== LoRA CONFIGURATION ===========================
# Adapter is merged into the base weights at GGUF export, so adapter size on disk is
# irrelevant. Full attention + MLP targets and a higher rank for stronger persona learning.
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0

# NOTE for Glimmer: "gate_proj" names TWO different Linears per layer - the SwiGLU
# mlp.gate_proj and the attention output gate self_attn.gate_proj
# (attn_output * sigmoid(gate_proj(hidden_states))). Both are inside the language model and
# both get adapted: 52 layers x (q, k, v, o, attention gate + mlp gate, up, down) = 416.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== RESPONSE MASKING ===========================
# Passed explicitly to train_on_responses_only, on purpose. Unsloth's auto-detection on this
# template returns '<|eot|><|start|>user<|message|>' and
# '<|eot|><|start|>assistant to=user<|message|>' - both carry the PREVIOUS turn's <|eot|>.
# The mask keeps labels from the end of the response marker up to the start of the next
# instruction marker, so with those markers the <|eot|> that closes every non-final
# assistant turn would be masked out and only the last turn would teach the model to stop.
# Dropping the leading <|eot|> keeps every assistant turn's <|eot|> in the loss. The
# trainer cell verifies this and raises if it does not hold.
RESPONSE_MASK_INSTRUCTION_PART = "<|start|>user<|message|>"
RESPONSE_MASK_RESPONSE_PART = "<|start|>assistant to=user<|message|>"

# =========================== INFERENCE TEST ===========================
# Meta's recommended sampling for Muse Glimmer (model card / generation_config.json).
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95
GEN_TOP_K = 64
GEN_MAX_NEW_TOKENS = 512

# =========================== GGUF EXPORT (Section 13, optional) ===========================
GGUF_QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

# ============================================================================
print("Configuration loaded (Muse Glimmer 30B 4-bit QLoRA)")
print(f"  Project root:    {PROJECT_ROOT}")
print(f"  HF hub cache:    {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"  Base model:      {BASE_LLM}")
print(f"  Model name:      {MODEL_NAME_BASE}")
print(f"  Input data:      {INPUT_DATA_FILE}")
print(f"  Output base:     {OUTPUT_BASE_DIR}")
print(f"  LoRA output:     {LORA_OUTPUT_DIR}")
print(f"  LoRA config:     r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training:        batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  SFT max examples: {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  Max seq length:  {MAX_SEQ_LENGTH} (dataset max measured at 3033 tokens)")
print(f"  Checkpoints:     every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Reasoning:       {REASONING_MARKER!r} (training, eval and GGUF default)")
print(f"  Persona prompts: extracted from JSONL at load time")

if not os.path.exists(INPUT_DATA_FILE):
    raise FileNotFoundError(f"Training data not found: {INPUT_DATA_FILE}")

Configuration loaded (Muse Glimmer 30B 4-bit QLoRA)
  Project root:    /workspace/training/biblical
  HF hub cache:    <default>
  Base model:      meta-models/Muse-Glimmer-30B
  Model name:      biblical_muse_glimmer_30b_unsloth_4bit
  Input data:      /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl
  Output base:     /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit
  LoRA output:     /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit/lora_adapters
  LoRA config:     r=32, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  Training:        batch=2, grad_accum=4, lr=0.0002
  SFT max examples: ALL
  Max seq length:  3072 (dataset max measured at 3033 tokens)
  Checkpoints:     every 50 steps, keep 3
  Reasoning:       'Reasoning strength: low.' (training, eval and GGUF default)
  Persona prompts: extracted from JSONL at load time


## 2. Environment Preparation

Safe to re-run. Order matters:

1. Verify the CUDA PyTorch build is intact (no side effects).
2. Core training packages, and remove the aarch64 `torchao` build that blocks PEFT's bnb 4-bit dispatcher.
3. `transformers` + `peft` from git main — **only if** the installed `transformers` has no `muse_glimmer` model package. The container as last used (transformers 5.18.0.dev0, peft 0.21.1.dev0) already has it, and a build that is known to work is not replaced.
4. Small utility packages.
5. Rebuild `causal_conv1d` from source if its CUDA extension is missing — the NGC image ships the Python package **without** it, which hard-crashes `import unsloth` itself, for any base model.
6. Import `unsloth` **before** `transformers` so its monkey-patches apply.

Glimmer is a plain attention model (39 sliding-window + 13 full-attention layers), so the `flash-linear-attention` step from the Qwen3.8 notebook is not needed and is not here.

**Restart the kernel after this cell, then resume from Cell 1.**

In [2]:
import os, sys, subprocess, importlib, importlib.util

def _pip(*args, env_extra=None):
    """Run pip against this kernel's interpreter; print output only on failure."""
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# --- 1. Verify the CUDA PyTorch build is intact ------------------------------
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "cpu" in torch.__version__:
        print("  CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")
print(f"  torch {torch.__version__} - CUDA {torch.version.cuda} - GPU: {torch.cuda.get_device_name(0)}")

# --- 2. Core training packages ------------------------------------------------
print("  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...")
_pip("install", "-q", "-U", "unsloth", "trl", "accelerate", "datasets", "bitsandbytes")

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel >=0.16 on PyPI, so
# uninstall - peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
_pip("uninstall", "-y", "-q", "torchao")

# --- 3. transformers + peft from git main (only if needed) --------------------
# Muse Glimmer needs the `muse_glimmer` model package in transformers. Look for it on disk
# rather than importing transformers here (unsloth must be imported first, step 6). If it is
# already present, leave transformers and peft exactly as installed.
_tf_spec = importlib.util.find_spec("transformers")
_tf_dir = os.path.dirname(_tf_spec.origin) if _tf_spec and _tf_spec.origin else None
if _tf_dir and os.path.isdir(os.path.join(_tf_dir, "models", "muse_glimmer")):
    print("  transformers already has muse_glimmer - leaving transformers + peft as installed")
else:
    print("  transformers has no muse_glimmer - installing transformers + peft from git main...")
    _pip("install", "-q", "-U", "git+https://github.com/huggingface/transformers.git")
    _pip("install", "-q", "-U", "git+https://github.com/huggingface/peft.git")

# --- 4. Small utility packages ------------------------------------------------
for _module, _install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(_module) is None:
        print(f"  Installing {_install_args[-1]}...")
        _pip(*_install_args)

# --- 5. Fix causal_conv1d -----------------------------------------------------
# The NGC image ships the causal_conv1d Python package WITHOUT its compiled CUDA
# extension (causal_conv1d_cuda). That hard-crashes any import that reaches the
# FalconH1 model inside transformers or unsloth, so it must be fixed BEFORE
# importing either.
#
# pip also caches a broken prebuilt aarch64 wheel, so --no-binary is required to
# force a source build, together with CAUSAL_CONV1D_FORCE_BUILD=TRUE. The first
# build takes a few minutes on aarch64; pip caches the result afterwards.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",   # DGX Spark GB10 = sm_120
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    # Clean up - don't leave partial imports that spoil unsloth's import order
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing - rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    _ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
               "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if _ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext - uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed - uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# --- 6. Import unsloth FIRST, then transformers -------------------------------
# Unsloth must be imported before transformers/trl/peft so its monkey-patches apply.
# Purge anything already loaded so the imports pick up the versions installed above.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
import peft
import trl

print()
print(f"  unsloth:       {unsloth.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  trl:           {trl.__version__}")
print(f"  causal_conv1d: {'OK' if _causal_ok else 'FALLBACK (torch path)'}")
print(f"  torchao:       {'PRESENT (should be absent)' if importlib.util.find_spec('torchao') else 'absent (correct)'}")
if importlib.util.find_spec("transformers.models.muse_glimmer") is None:
    raise RuntimeError(
        f"transformers {transformers.__version__} has no muse_glimmer model package - "
        "Muse Glimmer cannot be loaded. See the pip output above."
    )
print(f"  muse_glimmer:  present in transformers")
print()
print("Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 - CUDA 13.0 - GPU: NVIDIA GB10
  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...
  transformers already has muse_glimmer - leaving transformers + peft as installed
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

  unsloth:       2026.11.3
  transformers:  5.17.0
  peft:          0.21.1.dev0
  trl:           1.15.0
  causal_conv1d: OK
  torchao:       absent (correct)
  muse_glimmer:  present in transformers

Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).


## 3. Load Dataset

Load the combined multi-turn ShareGPT JSONL from the datagen notebooks.

- Already quality-filtered (no short answers, no AI refusals)
- Multi-turn: 4 QA pairs grouped per conversation by topic
- Each conversation has a persona-specific system prompt
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [3]:
import json, os, re, random
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL.
# This replaces hardcoded prompt dicts — prompts stay in sync with datagen automatically.
conversations = []
persona_system_prompts = {}   # persona_key -> full system prompt text
persona_counts = defaultdict(int)
persona_by_index = []         # parallel to `conversations`, used for the stratified cap below

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)

        # Extract persona name from "You are <Name>, ..." pattern
        sys_msg = conv["conversations"][0]["value"]
        match = re.match(r"You are (.+?),", sys_msg)
        if match:
            raw_name = match.group(1)
            # Normalize to snake_case key: lowercase, strip leading "the ", underscores for spaces
            key = raw_name.lower()
            key = re.sub(r"^the\s+", "", key)
            key = key.replace(" ", "_")
            persona_counts[key] += 1
            if key not in persona_system_prompts:
                persona_system_prompts[key] = sys_msg
        else:
            key = "unknown"
            print(f"  WARNING: could not extract persona from system prompt: {sys_msg[:80]}...")
        persona_by_index.append(key)

dataset = HFDataset.from_list(conversations)

# Apply SFT cap from config — stratified per persona, deterministic seed.
# Applied AFTER loading the JSONL, BEFORE quality validation (matches the PubMed SFT notebook).
#
# Cap logic follows docs/sft_notebook_guidelines.md ("Stratified Example Caps"):
#   1. WATER-FILLING. An even split takes min(target, available) per persona and drops the
#      shortfall of any undersized persona. The shortfall is redistributed over the personas
#      that still have rows, repeating until the cap is met or every pool is exhausted.
#   2. RANDOM SAMPLING. random.sample(pool, n), never pool[:n] - the JSONL is grouped by
#      topic, so a head slice biases a capped run toward whichever topics came first.
if SFT_MAX_EXAMPLES:
    if SFT_MAX_EXAMPLES < 0:
        raise ValueError(f"SFT_MAX_EXAMPLES must be 0 or positive, got {SFT_MAX_EXAMPLES}")
    random.seed(42)
    if len(dataset) > SFT_MAX_EXAMPLES:
        idx_by_persona = defaultdict(list)
        for i, p in enumerate(persona_by_index):
            idx_by_persona[p].append(i)

        # Water-fill: repeatedly split the remaining budget evenly across personas that
        # still have unallocated rows, capping each at what it actually has.
        personas_sorted = sorted(persona_counts.keys(), key=lambda p: -persona_counts[p])
        allocated = {p: 0 for p in personas_sorted}
        remaining = SFT_MAX_EXAMPLES
        open_personas = [p for p in personas_sorted if persona_counts[p] > 0]
        while remaining > 0 and open_personas:
            share = remaining // len(open_personas)
            if share == 0:
                # Fewer rows left than personas: hand out one each, largest pools first.
                for p in open_personas[:remaining]:
                    allocated[p] += 1
                    remaining -= 1
                break
            progressed = False
            for p in list(open_personas):
                take = min(share, persona_counts[p] - allocated[p])
                if take <= 0:
                    open_personas.remove(p)
                    continue
                allocated[p] += take
                remaining -= take
                progressed = True
                if allocated[p] >= persona_counts[p]:
                    open_personas.remove(p)
            if not progressed:
                break

        selected_idx = []
        for p in personas_sorted:
            pool = idx_by_persona[p]
            selected_idx.extend(random.sample(pool, allocated[p]))
        random.shuffle(selected_idx)

        even_target = SFT_MAX_EXAMPLES // len(personas_sorted)
        print(f"  Stratified cap: target {SFT_MAX_EXAMPLES} across {len(personas_sorted)} personas "
              f"(even split would be {even_target}/persona; shortfalls redistributed)")
        print(f"  Actually selected: {len(selected_idx)} rows")
        for p in personas_sorted:
            flag = "  <- pool exhausted" if allocated[p] >= persona_counts[p] else ""
            print(f"    {p:<25} {allocated[p]:>5} / {persona_counts[p]:<6} available{flag}")
        if len(selected_idx) < SFT_MAX_EXAMPLES:
            print(f"  NOTE: only {len(selected_idx)} rows available in total (< {SFT_MAX_EXAMPLES})")

        dataset = dataset.select(selected_idx)
        persona_counts = defaultdict(int, {p: allocated[p] for p in personas_sorted if allocated[p]})
    else:
        print(f"  SFT_MAX_EXAMPLES={SFT_MAX_EXAMPLES}, but only {len(dataset)} examples available; using all")

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(persona_counts)} personas")
print(f"Extracted {len(persona_system_prompts)} unique system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-persona breakdown:")
for p, c in sorted(persona_counts.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d} conversations")

# Flag severe imbalance — a persona with a tiny share will not hold its voice after training.
_total = sum(persona_counts.values())
_weak = [(p, c) for p, c in persona_counts.items() if c / _total < 0.05]
if _weak:
    print(f"\n  WARNING: personas under 5% of the training mix — expect weak voice separation:")
    for p, c in sorted(_weak, key=lambda x: x[1]):
        print(f"    {p:<25} {c:>5} ({100 * c / _total:.1f}%)")

# Show a sample prompt to verify extraction
sample_key = next(iter(persona_system_prompts))
print(f"\n--- Sample extracted prompt ({sample_key}, first 200 chars) ---")
print(f"  {persona_system_prompts[sample_key][:200]}...")

LOADING COMBINED SHAREGPT DATA
  File: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl

Total dataset: 4352 multi-turn conversations across 26 personas
Extracted 26 unique system prompts from JSONL
Columns: ['conversations', 'data_type']

Per-persona breakdown:
  moses                  785 conversations
  jeremiah               381 conversations
  paul                   377 conversations
  david                  350 conversations
  ezekiel                341 conversations
  isaiah                 332 conversations
  solomon                254 conversations
  job                    244 conversations
  daniel                 239 conversations
  peter                  148 conversations
  zechariah              145 conversations
  hosea                  107 conversations
  amos                    92 conversations
  joshua                  88 conversations
  micah                   73 conversations
  apostle_john            71 co

## 4. Validate & Summarize Dataset

Datagen data is already clean (no artifacts to strip). Verify data quality and show persona distribution.

In [4]:
bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count >=3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (should match extracted count: {len(persona_system_prompts)})")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\nBad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\nFiltering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Persona distribution
print(f"\nPERSONA DISTRIBUTION:")
max_name_len = max(len(n) for n in persona_counts)
for name, count in sorted(persona_counts.items(), key=lambda x: -x[1]):
    bar = "#" * (count // 50)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(persona_counts.values()):>5}")

# Show voice differentiation — first response from different personas
print(f"\nVOICE SAMPLES (first ~100 chars of response):")
seen_personas = set()
for example in dataset:
    system = example["conversations"][0]["value"]
    name_part = system.split(",")[0].replace("You are ", "")
    if name_part not in seen_personas and len(seen_personas) < 4:
        response_start = example["conversations"][2]["value"][:100]
        print(f"  {name_part}: \"{response_start}...\"")
        seen_personas.add(name_part)

print(f"\nDataset validated and ready for training")

DATA QUALITY CHECK
  Total examples: 4352
  Bad structure: 0
  Empty responses: 0
  Unique system prompts: 26 (should match extracted count: 26)
  Turn distribution: {3: 1741, 5: 91, 7: 541, 9: 1979}

PERSONA DISTRIBUTION:
  moses          785  ###############
  jeremiah       381  #######
  paul           377  #######
  david          350  #######
  ezekiel        341  ######
  isaiah         332  ######
  solomon        254  #####
  job            244  ####
  daniel         239  ####
  peter          148  ##
  zechariah      145  ##
  hosea          107  ##
  amos            92  #
  joshua          88  #
  micah           73  #
  apostle_john    71  #
  james           54  #
  malachi         44  
  joel            44  
  zephaniah       37  
  habakkuk        32  
  jonah           29  
  nahum           27  
  haggai          24  
  obadiah         19  
  jude            15  
  TOTAL         4352

VOICE SAMPLES (first ~100 chars of response):
  Daniel: "Four is the number that stay

## 5. Load Model & Tokenizer (4-bit)

Loads Meta's `meta-models/Muse-Glimmer-30B` (bf16) and quantizes it to bnb 4-bit on the fly. `use_exact_model_name=True` keeps Unsloth's mapper from substituting its own `-unsloth-bnb-4bit` re-pack; the `Resolved to:` line below must print the Meta name.

**First run downloads 59.6 GB** into the shared HF cache. Subsequent runs reuse it.

**Memory:** stop any serving container (`vllm-*`, `sglang-*`) first — host and device share one pool on this machine.

In [5]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and the broken
# torch.compile path (docs/dgx_spark_gb10_quirks.md section 4). Must be set BEFORE
# `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from collections import Counter
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_exact_model_name=True,   # load Meta's repo as-is; no mapper swap to unsloth's bnb-4bit re-pack
)
_resolved = getattr(model.config, "_name_or_path", "")
if "unsloth" in _resolved.lower() or BASE_LLM.split("/")[-1] not in _resolved:
    raise RuntimeError(f"Expected the Meta repo, but Unsloth loaded: {_resolved!r}")


# ---- Multimodal base: unwrap the Processor -------------------------------------
# This base is a *ForConditionalGeneration checkpoint, so Unsloth loads it via
# AutoModelForVision2Seq and returns a multimodal Processor, not a tokenizer.
# Unwrap it and use the inner tokenizer for the rest of the notebook:
#   - TRL takes its text path instead of calling process_row(), which expects images.
#   - Trainer._get_train_sampler reads processing_class.model_input_names[0]; on a
#     Processor that is "pixel_values", not "input_ids".
#   - A Processor's first positional parameter is `images`, so tokenizer("text") binds
#     to the wrong argument - always call it as tokenizer(text=...).
# `processor` is kept so the adapter directory is saved with the processor files too.
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor - text-only SFT mode)")

# The chat template must be on the object that renders the training text.
if not getattr(tokenizer, "chat_template", None):
    if processor is not None and getattr(processor, "chat_template", None):
        tokenizer.chat_template = processor.chat_template
        print("  (Copied chat template from Processor to tokenizer)")
    else:
        raise RuntimeError("Neither the tokenizer nor the Processor carries a chat template.")

# Glimmer ships a dedicated pad token (<|finetune_right_pad|>); the fallback is defensive.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# ---- Base model inspection (docs/multimodal_and_hybrid_base_models.md checklist) ----
_text_cfg = getattr(model.config, "text_config", None)
_layer_types = dict(Counter(getattr(_text_cfg, "layer_types", None) or []))
_arch = type(model).__name__

print(f"Model loaded: {BASE_LLM}")
print(f"  Resolved to:   {getattr(model.config, '_name_or_path', 'unknown')}")
print(f"  Class:         {_arch}")
print(f"  Precision:     4-bit (QLoRA)")
print(f"  Max seq len:   {MAX_SEQ_LENGTH}")
print(f"  Vocab size:    {getattr(tokenizer, 'vocab_size', 'unknown')}")
print(f"  Text layers:   {getattr(_text_cfg, 'num_hidden_layers', 'unknown')}  {_layer_types}")
print(f"  Vision tower:  {'present (kept frozen)' if getattr(model.config, 'vision_config', None) is not None else 'absent'}")
print(f"  BOS / EOS / pad: {tokenizer.bos_token!r} / {tokenizer.eos_token!r} / {tokenizer.pad_token!r}")
print(f"  Attn impl:     {getattr(model.config, '_attn_implementation', 'unknown')}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

if any("linear" in str(t) for t in _layer_types):
    raise RuntimeError(
        f"Unexpected linear-attention layers in this base ({_layer_types}). This notebook "
        "assumes a plain attention model - see docs/multimodal_and_hybrid_base_models.md."
    )

==((====))==  Unsloth 2026.11.3: Fast Muse_Glimmer patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1436 [00:00<?, ?it/s]

  (Extracted tokenizer from Processor - text-only SFT mode)
Model loaded: meta-models/Muse-Glimmer-30B
  Resolved to:   meta-models/Muse-Glimmer-30B
  Class:         MuseGlimmerForConditionalGeneration
  Precision:     4-bit (QLoRA)
  Max seq len:   3072
  Vocab size:    200000
  Text layers:   52  {'sliding_attention': 39, 'full_attention': 13}
  Vision tower:  present (kept frozen)
  BOS / EOS / pad: '<|begin_of_text|>' / '<|end_of_text|>' / '<|finetune_right_pad|>'
  Attn impl:     flash_attention_2
  GPU allocated: 22.1 GB


## 6. Format Dataset for Chat Template

Apply Glimmer's native chat template to each conversation and build the final training dataset.

Rendered with `add_generation_prompt=False` and `reasoning_strength=REASONING_STRENGTH`. A rendered conversation looks like:

```
<|begin_of_text|><|start|>system<|message|>{persona prompt}

Reasoning strength: low.

# Valid recipients: "self", "user".<|eot|><|start|>user<|message|>{question}<|eot|><|start|>assistant to=user<|message|>{answer}<|eot|>...
```

Four properties are checked on **every** row, and each one raises: the expected reasoning line is present (and no other level), there is no reasoning channel (`to=self`), every row starts with the BOS token and ends with `<|eot|>`, and no row exceeds `MAX_SEQ_LENGTH` tokens.

In [6]:
# Standardize ShareGPT rows, then format with the Glimmer tokenizer chat template
from unsloth.chat_templates import standardize_sharegpt

original_dataset_len = len(dataset)

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
    reasoning_strength=REASONING_STRENGTH,   # same value the inference cells pass
)

# ---- Guard 1: reasoning line. The template always writes one; it must be OURS. ----
_missing = sum(1 for t in formatted_texts if REASONING_MARKER not in t)
_other = sum(
    1 for t in formatted_texts
    if any(f"Reasoning strength: {lvl}." in t
           for lvl in ("low", "medium", "high", "xhigh") if lvl != REASONING_STRENGTH)
)
if _missing or _other:
    raise RuntimeError(
        f"Reasoning line check failed: {_missing} row(s) lack {REASONING_MARKER!r}, {_other} "
        "row(s) carry a different level. The reasoning_strength template variable did not "
        "take effect - do not start training."
    )

# ---- Guard 2: no reasoning channel, correct framing. ----
_to_self = sum(1 for t in formatted_texts if "<|start|>assistant to=self" in t)
_bad_frame = sum(
    1 for t in formatted_texts
    if not t.startswith(tokenizer.bos_token) or not t.endswith("<|eot|>")
    or "<|start|>assistant to=user<|message|>" not in t
)
if _to_self or _bad_frame:
    raise RuntimeError(
        f"Template framing check failed: {_to_self} row(s) contain a to=self reasoning turn, "
        f"{_bad_frame} row(s) do not start with BOS / end with <|eot|> / contain a to=user "
        "answer. The chat template is not the one this notebook was written against."
    )

# ---- Guard 3: nothing may be truncated. ----
# add_special_tokens=False: the template already rendered <|begin_of_text|>, and this
# tokenizer prepends another one on a plain encode.
_token_lens = [len(ids) for ids in tokenizer(text=formatted_texts, add_special_tokens=False)["input_ids"]]
_too_long = sum(1 for n in _token_lens if n > MAX_SEQ_LENGTH)
if _too_long:
    raise RuntimeError(
        f"{_too_long} conversation(s) exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} tokens "
        f"(longest is {max(_token_lens)}). Truncation would cut the final answer and its "
        "<|eot|>. Raise MAX_SEQ_LENGTH in Cell 1."
    )

# Build final dataset
import pandas as pd
from datasets import Dataset as HFDataset

dataset = HFDataset.from_pandas(pd.DataFrame({"text": formatted_texts}))

# Filter out empty examples and shuffle. SFT_MAX_EXAMPLES is already applied - the
# stratified per-persona cap runs in the data loading cell, before this formatting step.
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
filtered_dataset_len = len(dataset)
dataset = dataset.shuffle(seed=42)

_sorted_lens = sorted(_token_lens)
print(f"--- Sample formatted text (first 500 chars) ---")
print(dataset[0]["text"][:500])
print(f"\n--- Sample formatted text (last 200 chars) ---")
print(dataset[0]["text"][-200:])
print(f"\nDataset formatted: {len(dataset)} examples")
print(f"  Original conversations: {original_dataset_len}")
print(f"  After empty filtering:  {filtered_dataset_len}")
print(f"  SFT cap:                {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  Token lengths:          min {_sorted_lens[0]}, median {_sorted_lens[len(_sorted_lens)//2]}, "
      f"max {_sorted_lens[-1]} (limit {MAX_SEQ_LENGTH})")
print(f"  Reasoning line:         {REASONING_MARKER!r} on every row, no to=self turns")

Unsloth: Standardizing formats (num_proc=8):   0%|          | 0/4352 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4352 [00:00<?, ? examples/s]

--- Sample formatted text (first 500 chars) ---
<|begin_of_text|><|start|>system<|message|>You are David, the king of Israel, once a shepherd boy and warrior, the sweet psalmist who poured out your heart in song, a man after God's own heart who knew both triumph and deep sin.

YOUR DISTINCTIVE VOICE: Poetic, emotional, lyrical. Psalm cadence — parallelism, repetition, selah-like pauses. Shifts between ecstatic praise, raw anguish, and intimate confession. Uses nature imagery (shepherd, waters, mountains). Addresses God directly ('O LORD'). Un

--- Sample formatted text (last 200 chars) ---
e not thy commandments from me.

My soul breaketh for the longing that it hath unto thy judgments at all times.

Thou hast rebuked the proud that are cursed, which do err from thy commandments.<|eot|>

Dataset formatted: 4352 examples
  Original conversations: 4352
  After empty filtering:  4352
  SFT cap:                ALL
  Token lengths:          min 501, median 1588, max 3033 (limit 3072)
  Reas

## 7. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See Cell 1 config for module options.

### Freezing the perception encoder — deliberately, not by accident

`meta-models/Muse-Glimmer-30B` is a multimodal checkpoint: alongside the 52-layer language model it carries a 50-layer ViT perception encoder (`model.vision_tower`), a two-layer `model.vision_adapter` and a `model.vision_projection`. This notebook trains a **text-only** persona LoRA, so none of that should receive an adapter.

That does **not** happen for free. PEFT matches a `target_modules` list like `["q_proj", "k_proj", ...]` by **suffix, across the entire module tree** — it has no notion of "the language model". On Glimmer the collision is real: the vision tower's attention uses `attn.q_proj`, `attn.k_proj` and `attn.v_proj`, exactly the leaf names in the list. A bare list would attach LoRA to 150 vision modules (50 layers × 3). Its `attn.proj`, `mlp.fc1` and `mlp.fc2` do not collide.

Two adapters in this workspace shipped with vision and audio LoRA tensors attached this way. They were harmless in effect — `lora_B` is zero-initialised and text batches never invoke those towers — but they were 36–42% dead weight, and **vLLM refuses to load an adapter carrying vision-layer LoRA**.

The `finetune_*` flags below make the scoping explicit (Unsloth then matches only module paths containing `language`/`text`), and the assertion after the call proves it held rather than assuming it.

Glimmer has no MTP head. Its speculative-decoding companion is a separate repo (`meta-models/Muse-Glimmer-30B-assistant`, a DFlash drafter) and is not part of this model.

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    # ---- TOWER FREEZE: deliberate, not incidental ----------------------------------
    # These four flags are load-bearing. Unsloth routes an explicit target_modules LIST
    # through its language-scoped regex ONLY when at least one of them is False; all four
    # default to True, so a bare list goes straight to PEFT, which matches by SUFFIX
    # across the WHOLE module tree. finetune_vision_layers=False is what confines the
    # match to the language model: the scoped regex requires a `language`/`text` path
    # segment, which model.language_model.* has and model.vision_tower.* does not.
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # Deliberately True, NOT "unsloth". The "unsloth" path copies saved activations into
    # pinned CPU buffers that only ever grow - unsloth_zoo/gradient_checkpointing.py
    # resizes them up and never down. On GB10 host and device share one 128 GB pool, so
    # that offload frees no capacity while ratcheting unreclaimable pinned pages upward
    # for hours. True uses standard recompute checkpointing with no host copies.
    use_gradient_checkpointing=True,
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES})")
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# ============ VERIFY THE ADAPTER SCOPE AND THE TOWER FREEZE ============
# This is a TEXT-ONLY fine-tune on a multimodal base. Two properties must hold, and they
# fail in different ways, so both are checked:
#
#   1. WHERE ADAPTERS ATTACHED. PEFT matches a target_modules LIST by suffix across the
#      whole module tree - it has no concept of "the language model". Any tower reusing
#      the language model's leaf names (q_proj, gate_proj, ...) collects adapters too.
#   2. WHAT IS TRAINABLE. Under QLoRA base weights are frozen 4-bit and only LoRA A/B
#      carry gradients, so the towers are frozen by construction. Assert it anyway - an
#      assumption that is never checked is an assumption that eventually breaks.
#
# See docs/multimodal_and_hybrid_base_models.md.
from collections import Counter as _Counter


def _zone_of(param_name):
    """Return the non-language sub-model a parameter belongs to, or None."""
    n = param_name.replace("base_model.model.", "", 1)
    if "vision_tower" in n or ".visual." in n or n.startswith("visual."):
        return "vision tower"
    if "vision_adapter" in n or "vision_projection" in n or "multi_modal_projector" in n:
        return "vision projector"
    if "audio_tower" in n:
        return "audio tower"
    if n.startswith("mtp.") or ".mtp." in n:
        return "mtp head"
    return None


_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
_stray = [n for n in _adapted if _zone_of(n)]
if _stray:
    raise RuntimeError(
        f"LoRA attached to {len(_stray)} module(s) outside the language model "
        f"(e.g. {_stray[:3]}). The finetune_* scoping flags did not take effect. "
        "Do not train - vLLM refuses adapters carrying vision-layer LoRA."
    )

_unfrozen, _zone_params = {}, _Counter()
for _n, _p in model.named_parameters():
    _z = _zone_of(_n)
    if _z is None:
        continue
    _zone_params[_z] += 1
    if _p.requires_grad:
        _unfrozen.setdefault(_z, []).append(_n)

_outside = [n for n in _adapted if "language_model" not in n]
if not _adapted or _outside:
    raise RuntimeError(
        f"LoRA scope is wrong: {len(_adapted)} module(s) adapted, {len(_outside)} of them "
        f"outside model.language_model (e.g. {_outside[:3]}). Do not train."
    )

_n_layers = model.config.text_config.num_hidden_layers
_expected = _n_layers * 8   # q, k, v, o, attention gate_proj + mlp gate, up, down
_by_leaf = _Counter(".".join(n.split(".")[-2:]) for n in _adapted)
print(f"  LoRA adapted {len(_adapted)} modules, all inside the language model "
      f"(expected {_expected} = {_n_layers} layers x 8).")
for _leaf, _c in sorted(_by_leaf.items()):
    print(f"    {_leaf:<22} {_c:>4}")
if len(_adapted) != _expected:
    print(f"  NOTE: adapted-module count differs from the expected {_expected}. Compare the "
          "per-leaf breakdown above with LORA_TARGET_MODULES before a long run.")
for _z, _c in sorted(_zone_params.items()):
    _bad = len(_unfrozen.get(_z, []))
    print(f"  {_z:<14} {_c:>5} params  "
          f"{'FROZEN' if not _bad else str(_bad) + ' TRAINABLE - BAD'}")
if _unfrozen:
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} vision/projector parameter(s) are trainable "
        f"(e.g. {[n for v in _unfrozen.values() for n in v][:3]}). This is a text-only "
        "fine-tune - they must stay frozen."
    )

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
LoRA adapters added (r=32, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
Trainable parameters: 209,584,128 / 17,369,744,384 (1.21%)
  LoRA adapted 416 modules, all inside the language model (expected 416 = 52 layers x 8).
    mlp.down_proj            52
    mlp.gate_proj            52
    mlp.up_proj              52
    self_attn.gate_proj      52
    self_attn.k_proj         52
    self_attn.o_proj         52
    self_attn.q_proj         52
    self_attn.v_proj         52
  vision projector     3 params  FROZEN
  vision tower     806 params  FROZEN


## 8. Trainer Setup

- 1 epoch — gently teaches persona-switching without degrading base capabilities
- Each example has a persona-specific system prompt so the model learns distinct voices
- Loss is computed on assistant answers only, including the `<|eot|>` that closes each one

No `PYTORCH_CUDA_ALLOC_CONF` and no memory-fraction cap are set, per `docs/dgx_spark_gb10_quirks.md`. The cleanup callback prints allocated/reserved memory every `CLEANUP_STEPS` steps: peak allocation flat while *reserved* climbs is the signature of allocator fragmentation.

In [8]:
import gc
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig


class PeriodicMemoryCleanup(TrainerCallback):
    """Collect dead Python objects, return cached CUDA blocks, and report memory.

    Runs every `every` optimizer steps. This does not repair a leak - it releases
    cached/fragmented blocks that would otherwise sit reserved but unused, which
    matters on GB10 where host and device draw from the same 128 GB pool. The printed
    `reserved` figure is what to watch: it should plateau, not climb step after step.
    """

    def __init__(self, every=50):
        self.every = max(1, int(every))

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
            print(f"  [mem] step {state.global_step}: "
                  f"allocated {torch.cuda.memory_allocated() / 2**30:.1f} GiB, "
                  f"reserved {torch.cuda.memory_reserved() / 2**30:.1f} GiB, "
                  f"peak allocated {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
        return control


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        # One conversation per row, as in the confirmed Qwen3.8 runs. Packing would also
        # concatenate conversations with different persona system prompts into one sequence.
        packing=False,
        # No train_sampling_strategy="group_by_length": see the Qwen3.8 notebook and
        # docs/multimodal_and_hybrid_base_models.md section 2. Default random sampler;
        # batches pad to their own longest member.
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
    ),
    callbacks=[PeriodicMemoryCleanup(CLEANUP_STEPS)],
)

# ===================== TOKENIZATION SANITY (raises) =====================
# The formatted text already starts with <|begin_of_text|>, and this tokenizer prepends
# another one on a plain encode. Unsloth's dataset preparation detects that and tokenizes
# with add_special_tokens=False; TRL's own preparation would additionally append the
# tokenizer EOS (<|end_of_text|>) after the final <|eot|>. Check what actually got built.
import numpy as np

_bos_id = tokenizer.bos_token_id
_eot_id = tokenizer.convert_tokens_to_ids("<|eot|>")
_probe_ids = [list(x) for x in trainer.train_dataset[:8]["input_ids"]]
for _ids in _probe_ids:
    if _ids[0] != _bos_id or _ids.count(_bos_id) != 1:
        raise RuntimeError(
            f"Tokenized training row has {_ids.count(_bos_id)} BOS token(s), first id {_ids[0]} "
            f"(BOS is {_bos_id}). Expected exactly one, at position 0. Do not start training."
        )
    if _ids[-1] != _eot_id:
        raise RuntimeError(
            f"Tokenized training row ends with {tokenizer.convert_ids_to_tokens(_ids[-1])!r}, "
            "not '<|eot|>'. Something appended a token after the final turn (or the row was "
            "truncated). Do not start training."
        )

# ===================== TRAIN ON RESPONSES ONLY =====================
# Mask prompt tokens to -100 so loss is computed ONLY on assistant turns.
# Without this the loss averages in the persona system prompt and the user turns -
# tokens the base model already predicts near-perfectly and that repeat across all
# 4,352 examples. They pin the average near their own ~0 loss and bury the signal
# from the response tokens.
#
# Markers are passed explicitly here - see RESPONSE MASKING in Cell 1 for why Unsloth's
# auto-detected markers are not used on this template.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part=RESPONSE_MASK_INSTRUCTION_PART,
    response_part=RESPONSE_MASK_RESPONSE_PART,
)

# Verify the mask. Three ways it can be wrong, all silent for the whole run otherwise:
#   - every label is -100 (markers did not match): the run trains on nothing;
#   - prompt text leaked into the labels;
#   - the <|eot|> closing an assistant turn is masked: the model is not taught to stop.
_probe = trainer.train_dataset[:8]
_kept = _total = 0
for _ids, _labels in zip(_probe["input_ids"], _probe["labels"]):
    _ids, _labels = list(_ids), np.array(_labels)
    _kept_ids = [int(t) for t in _labels if t != -100]
    _kept += len(_kept_ids)
    _total += len(_labels)
    if not _kept_ids:
        raise RuntimeError(
            "train_on_responses_only masked EVERY token of a row - the instruction/response "
            "markers did not match the chat template. Do not start training."
        )
    _kept_text = tokenizer.decode(_kept_ids)
    if "<|start|>" in _kept_text or "Reasoning strength:" in _kept_text:
        raise RuntimeError(
            "Response mask leaked prompt text into the labels (a turn header or the system "
            f"prompt is being trained on). Kept text starts: {_kept_text[:200]!r}"
        )
    _n_answers = tokenizer.decode(_ids).count(RESPONSE_MASK_RESPONSE_PART)
    _n_eot_kept = _kept_ids.count(_eot_id)
    if _n_eot_kept != _n_answers:
        raise RuntimeError(
            f"Row has {_n_answers} assistant turn(s) but only {_n_eot_kept} <|eot|> label(s) "
            "survive the mask. Every assistant turn must keep its closing <|eot|>."
        )
print(f"Response masking OK: {_kept:,}/{_total:,} label tokens kept "
      f"({100 * _kept / _total:.1f}%) across 8 sample sequences; every assistant turn keeps its <|eot|>")

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
print(f"Trainer configured")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: disabled")
print(f"  Checkpoints: every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Memory cleanup + report: every {CLEANUP_STEPS} steps")
print(f"  Loss computed on: assistant responses only (prompt masked to -100)")
print(f"  Dataset: {len(dataset)} examples")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/4352 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/4352 [00:00<?, ? examples/s]

Response masking OK: 7,054/11,009 label tokens kept (64.1%) across 8 sample sequences; every assistant turn keeps its <|eot|>
Trainer configured
  Effective batch size: 2 x 4 = 8
  Epochs: 1
  LR: 0.0002
  Packing: disabled
  Checkpoints: every 50 steps, keep 3
  Memory cleanup + report: every 50 steps
  Loss computed on: assistant responses only (prompt masked to -100)
  Dataset: 4352 examples
  Precision: bf16


## 9. Train

In [ ]:
# Start training. Auto-resumes from the newest checkpoint in OUTPUT_DIR_ADAPTERS if one
# exists, so an interrupted run continues instead of restarting from step 0.
import os
from transformers.trainer_utils import get_last_checkpoint

_ckpt_dir = trainer.args.output_dir
last_checkpoint = get_last_checkpoint(_ckpt_dir) if os.path.isdir(_ckpt_dir) else None

if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found - starting from scratch.")
    result = trainer.train()

print("\nTraining complete")
print(f"  Final loss:  {result.training_loss:.4f}")
print(f"  Total steps: {result.global_step}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 200001, 'bos_token_id': 200000}.


No checkpoint found - starting from scratch.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,352 | Num Epochs = 1 | Total steps = 544
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 209,584,128 of 29,986,210,816 (0.70% trained)


Step,Training Loss
5,1.686949
10,1.533159
15,1.284482
20,1.320326
25,1.275873
30,1.476341
35,1.482610
40,1.343641
45,1.386827
50,1.188409


  [mem] step 50: allocated 21.8 GiB, reserved 22.4 GiB, peak allocated 27.5 GiB


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit/train/checkpoint-50/tokenizer_config.json.


  [mem] step 100: allocated 21.8 GiB, reserved 22.4 GiB, peak allocated 28.1 GiB


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit/train/checkpoint-100/tokenizer_config.json.


## 10. Save LoRA Adapters

Save the trained LoRA adapters, the persona system prompts, and a `complete.json` sentinel. The adapters load on the Muse Glimmer 30B base via PEFT; Section 13 merges them into the bf16 base for GGUF.

In [ ]:
from pathlib import Path
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
# Save the Processor when we unwrapped one: that writes processor_config.json alongside
# the tokenizer files and chat_template.jinja, a superset of tokenizer.save_pretrained().
(processor or tokenizer).save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

# Machine-readable completion sentinel (docs/README.md: "Final model-producing
# notebooks must write a machine-readable completion sentinel").
from datetime import datetime, timezone
sentinel = {
    "status": "complete",
    "stage": "sft",
    "model_name": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "input_data_file": INPUT_DATA_FILE,
    "output_dir": OUTPUT_DIR_ADAPTERS,
    "lora_output_dir": LORA_OUTPUT_DIR,
    "global_step": int(result.global_step),
    "final_loss": float(result.training_loss),
    "num_train_epochs": TARGET_EPOCHS,
    "sft_max_examples": SFT_MAX_EXAMPLES,
    "train_examples": len(dataset),
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "adapted_modules": len(_adapted),
    "reasoning_strength": REASONING_STRENGTH,
    "response_mask_parts": [RESPONSE_MASK_INSTRUCTION_PART, RESPONSE_MASK_RESPONSE_PART],
    "personas": sorted(persona_system_prompts),
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
with open(f"{LORA_OUTPUT_DIR}/complete.json", "w") as f:
    json.dump(sentinel, f, indent=2)

example_persona = next(iter(persona_system_prompts), "amos")

print(f"\nLoRA adapters saved")
print(f"  Sentinel:       {LORA_OUTPUT_DIR}/complete.json (step {result.global_step}, "
      f"loss {result.training_loss:.4f})")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["{example_persona}"]  # or any persona key')

## 11. Test Inference

Quick smoke test with a few personas using their extracted system prompts. Each persona should respond in its distinctive voice.

The generation prompt ends at `<|start|>assistant`, so the model chooses the channel itself and the streamed output **begins with the channel header**:

- ` to=user<|message|>...` — a direct answer. This is what training taught.
- ` to=self<|message|>...<|eom|>` followed by a `to=user` turn — the model reasoned first.

Special tokens are left visible on purpose so the channel can be seen.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Pick up to 4 personas to test
test_personas = list(persona_system_prompts.keys())[:4]

print(f"INFERENCE TEST - {len(test_personas)} PERSONAS (reasoning strength: {REASONING_STRENGTH})\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    # reasoning_strength must match training. Omitting it renders the template default
    # ('high'), a system-prompt line the adapter was never trained under.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        reasoning_strength=REASONING_STRENGTH,
    )
    # add_special_tokens=False: the template already rendered <|begin_of_text|> and this
    # tokenizer would prepend a second one. Passed by keyword because a Processor's first
    # positional parameter is `images`, not `text`.
    inputs = tokenizer(text=text, return_tensors="pt", add_special_tokens=False).to(model.device)

    print(f"{'='*60}")
    print(f"  PERSONA: {persona_key.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        top_k=GEN_TOP_K,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

## 12. Verify Adapter (Reload from Disk)

Final validation: load the adapter cold from disk to confirm it is self-contained and portable.

In [ ]:
# Clean up training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_exact_model_name=True,   # the adapter's base_model_name_or_path is the Meta repo; keep it
)
FastLanguageModel.for_inference(model2)
# Unwrap the Processor, exactly as in the model-load cell.
tok2 = getattr(tokenizer2, "tokenizer", tokenizer2)

# Reload saved system prompts
import json
from pathlib import Path
with open(f"{LORA_OUTPUT_DIR}/persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

# Test with first persona
test_key = list(reloaded_prompts.keys())[0]
test_prompt_text = reloaded_prompts[test_key]

messages = [
    {"role": "system", "content": test_prompt_text},
    {"role": "user", "content": TEST_PROMPT},
]

text = tok2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    reasoning_strength=REASONING_STRENGTH,
)
if REASONING_MARKER not in text:
    raise RuntimeError(f"The reloaded chat template did not render {REASONING_MARKER!r}.")
inputs = tok2(text=text, return_tensors="pt", add_special_tokens=False).to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=GEN_MAX_NEW_TOKENS,
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    top_k=GEN_TOP_K,
    do_sample=True,
)

# Special tokens kept so the channel header (to=user / to=self) is visible.
response = tok2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:700]}")
if response.lstrip().startswith("to=self"):
    print("\n  NOTE: the model opened a reasoning turn (to=self) before answering. Training "
          "data had none; check the loss curve and the reasoning line in the prompt.")
print(f"\nAdapter loads cleanly from disk.")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, tok2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

## 13. Export Merged Model to GGUF (Mac / Metal, llama.cpp) — optional

Merge the LoRA adapter into the bf16 base and export GGUF, for llama.cpp-based runners on Apple Silicon (llama.cpp, LM Studio, Ollama). This produces a **text-only** GGUF; the perception encoder (`mmproj`) is not part of this export.

**Skip these cells unless you want GGUF** — the adapter saved in Section 10 is the artifact for PEFT-based serving.

Three things specific to Glimmer:

1. **llama.cpp must know the architecture.** Upstream llama.cpp converts Glimmer through `conversion/muse_glimmer.py`. The checkout this container had at `/root/.unsloth/llama.cpp` on 2026-11-09 predates that (its converter is from April 2026 and has no Glimmer support). The export cell checks the checkout first and raises with instructions instead of failing halfway through a 56 GB merge. The runner on the Mac needs a llama.cpp build that knows Glimmer as well.
2. **Download and memory.** This reloads the base at `load_in_4bit=False`: the bf16 Meta repo `meta-models/Muse-Glimmer-30B` (already in the cache from training, ~56 GiB in memory) plus merge overhead. Run it in a fresh kernel (Cell 1, then this section) with serving containers stopped. No memory-fraction cap is set, per `docs/dgx_spark_gb10_quirks.md`.
3. **Stop token and reasoning default.** Unlike the Qwen3.8 export, EOS is **not** overridden here: upstream's Glimmer converter writes `<|eot|>` into the GGUF as the end-of-turn token itself, and the exported file is checked for it afterwards. What *is* changed is the chat template embedded in the GGUF: its default reasoning strength is set from `'high'` to `REASONING_STRENGTH`, because a Mac runner renders the embedded template with no `reasoning_strength` variable and would otherwise prompt the model with a level it was not trained under.

In [ ]:
# Compatibility shim for PEFT + GPTQModel during GGUF export.
# PEFT imports GPTQModel's older AWQ class name even when this adapter is not AWQ.
try:
    import gptqmodel.nn_modules.qlinear.gemm_awq as _gemm_awq
    if not hasattr(_gemm_awq, "AwqGEMMQuantLinear") and hasattr(_gemm_awq, "AwqGEMMLinear"):
        _gemm_awq.AwqGEMMQuantLinear = _gemm_awq.AwqGEMMLinear
        print("Patched GPTQModel AWQ class alias for PEFT compatibility.")
except Exception as exc:
    print(f"GPTQModel AWQ compatibility shim skipped: {exc}")

In [ ]:
# Reload adapter fresh so we export from a clean state
import os
os.environ.setdefault("UNSLOTH_ENABLE_FLEX_ATTENTION", "0")
os.environ.setdefault("UNSLOTH_COMPILE_DISABLE", "1")

import gc, sys, torch
from pathlib import Path
from unsloth import FastLanguageModel
from unsloth_zoo.llama_cpp import LLAMA_CPP_DEFAULT_DIR

for _name in ("model2", "tokenizer2", "model_gguf", "tokenizer_gguf"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

# ---- Preflight: does the existing llama.cpp checkout know Muse Glimmer? ----------
# If there is no checkout, Unsloth clones current upstream, which does. An existing
# checkout is reused as-is, so a stale one must be caught before the merge.
_lcpp = Path(LLAMA_CPP_DEFAULT_DIR)
if _lcpp.exists():
    _new_layout = _lcpp / "conversion" / "muse_glimmer.py"
    _old_layout = _lcpp / "convert_hf_to_gguf.py"
    _knows_glimmer = _new_layout.exists() or (
        _old_layout.exists() and "MuseGlimmer" in _old_layout.read_text(errors="ignore")
    )
    if not _knows_glimmer:
        raise RuntimeError(
            f"The llama.cpp checkout at {_lcpp} has no Muse Glimmer converter "
            "(no conversion/muse_glimmer.py, and convert_hf_to_gguf.py does not mention "
            "MuseGlimmer). Move it aside so Unsloth clones and builds current upstream, e.g.\n"
            f"    mv {_lcpp} {_lcpp}.pre-glimmer\n"
            "then re-run this cell."
        )
    print(f"llama.cpp checkout at {_lcpp} has a Muse Glimmer converter.")
else:
    print(f"No llama.cpp checkout at {_lcpp} - Unsloth will clone and build upstream.")

Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter merges before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
    use_exact_model_name=True,   # merge onto Meta's bf16 repo, not a mapper substitute
)
_tok = getattr(tokenizer_gguf, "tokenizer", tokenizer_gguf)
_eot_id = _tok.convert_tokens_to_ids("<|eot|>")
if _eot_id is None or _eot_id == _tok.unk_token_id:
    raise RuntimeError(f"'<|eot|>' is not in the tokenizer vocab (got id={_eot_id})")

# ---- Bake the trained reasoning strength into the exported template's default ----
# The template reads:  reasoning_strength if ... and reasoning_strength else 'high'
# GGUF runners render the embedded template with no reasoning_strength variable.
_NEEDLE = "reasoning_strength else 'high'"
_BAKED = f"reasoning_strength else '{REASONING_STRENGTH}'"
if REASONING_STRENGTH != "high":
    _holders = [o for o in (tokenizer_gguf, _tok) if getattr(o, "chat_template", None)]
    if not _holders:
        raise RuntimeError("No chat template found on the reloaded tokenizer/processor.")
    for _o in {id(o): o for o in _holders}.values():
        if _o.chat_template.count(_NEEDLE) != 1:
            raise RuntimeError(
                f"Expected exactly one {_NEEDLE!r} in the chat template, found "
                f"{_o.chat_template.count(_NEEDLE)}. The template changed - do not export."
            )
        _o.chat_template = _o.chat_template.replace(_NEEDLE, _BAKED)
_render = _tok.apply_chat_template(
    [{"role": "system", "content": "x"}, {"role": "user", "content": "y"}], tokenize=False
)
if REASONING_MARKER not in _render:
    raise RuntimeError(f"Exported template default does not render {REASONING_MARKER!r}.")
print(f"Exported chat template defaults to {REASONING_MARKER!r}")

# Pass ALL quant methods in one call so the LoRA->bf16 merge happens ONCE
# and llama.cpp quantizes from that single merged file.
print(f"Exporting GGUF (single merge -> {len(GGUF_QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method=GGUF_QUANT_METHODS,
)

_ggufs = sorted(Path(GGUF_OUTPUT_DIR).rglob("*.gguf")) or sorted(Path(OUTPUT_BASE_DIR).rglob("*.gguf"))
if not _ggufs:
    raise RuntimeError(f"No .gguf files found under {OUTPUT_BASE_DIR} after export.")
print(f"\nGGUF export complete:")
for f in _ggufs:
    print(f"  {str(f):90s} {f.stat().st_size / 2**30:>7.1f} GiB")

# ---- Verify the exported metadata: stop token and embedded template ---------------
_check = min((f for f in _ggufs if "mmproj" not in f.name.lower()), key=lambda f: f.stat().st_size)
_meta = None
try:
    sys.path.insert(0, str(Path(LLAMA_CPP_DEFAULT_DIR) / "gguf-py"))
    from gguf import GGUFReader
    _reader = GGUFReader(str(_check))

    def _field(key):
        _f = _reader.get_field(key)
        return None if _f is None else _f.parts[_f.data[0]]

    _eot_f, _eos_f, _tmpl_f = (_field("tokenizer.ggml.eot_token_id"),
                               _field("tokenizer.ggml.eos_token_id"),
                               _field("tokenizer.chat_template"))
    _meta = {
        "eot": None if _eot_f is None else int(_eot_f[0]),
        "eos": None if _eos_f is None else int(_eos_f[0]),
        "template": None if _tmpl_f is None else bytes(_tmpl_f).decode("utf-8", "replace"),
    }
except Exception as exc:
    print(f"\nWARNING: could not read GGUF metadata from {_check.name} ({exc!r}). "
          "Stop token and embedded template are UNVERIFIED - check them on the Mac.")

if _meta is not None:
    print(f"\nGGUF metadata ({_check.name}): eos={_meta['eos']}, eot={_meta['eot']} (<|eot|> is {_eot_id})")
    if _eot_id not in (_meta["eot"], _meta["eos"]):
        raise RuntimeError(
            f"The GGUF does not stop on <|eot|> ({_eot_id}): eos={_meta['eos']}, eot={_meta['eot']}. "
            "The model would run on into the next turn. Do not ship these files."
        )
    if REASONING_STRENGTH != "high" and (_meta["template"] is None or _BAKED not in _meta["template"]):
        raise RuntimeError(
            "The chat template embedded in the GGUF does not carry the baked reasoning default "
            f"({_BAKED!r}). Runners would prompt with 'Reasoning strength: high.'"
        )
    print("  Stop token and embedded template verified.")

print("\nMac usage (llama.cpp / LM Studio / Ollama on Metal):")
print("  1. Copy the q4_k_m .gguf to the Mac (q5_k_m / q8_0 if memory allows).")
print("  2. The runner's llama.cpp build must support the muse-glimmer architecture.")
print("  3. Use the chat template embedded in the file (llama.cpp: --jinja). Do not substitute ChatML.")
print(f"  4. System prompt = one of persona_system_prompts.json; the template appends {REASONING_MARKER!r}")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()